In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import glob
import os
from datetime import datetime, timedelta
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap

Plot spatial maps of temperature with wind vectors on top + stress categories for UTCI10 and UTCI90 for two snapshots on 13 and 17 January 2017 at 15:00 for LCZ, WSF-MB and Geoscape - Figure 13

In [ ]:
#3x3 figure

#custom colorbar
# Define the temperature boundaries and corresponding colors
boundaries = [20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50]
colors = ['green', 'green', 'green', 'yellow', 'yellow', 'yellow', 'orange', 'orange', 'orange', 'red', 'red', 'red', 'darkred', 'darkred', 'darkred']

# Using ListedColormap for discrete colors
def create_utci_colormap_discrete():
    """Create a discrete colormap with sharp boundaries between colors"""
    cmap = mcolors.ListedColormap(colors)
    norm = mcolors.BoundaryNorm(boundaries, cmap.N)
    return cmap, norm
# Get the colormap
utci_cmap, utci_norm = create_utci_colormap_discrete()

# Define location points
loc1_lat, loc1_lon = -33.668, 150.932
loc2_lat, loc2_lon = -33.812, 151.009

# Directory paths for different scenarios
paths = {
    "LCZ": "/g/data/fy29/mf9078/run_WRF/etamodified/output/1out/selected/",
    "WSF-MB": "/g/data/fy29/mf9078/run_WRF/etamodified/output/3out/selected/",
    "Geoscape": "/g/data/fy29/mf9078/run_WRF/etamodified/output/4out/selected/"
}

# Domain boundaries
lat_min, lat_max = -34.14, -33.55
lon_min, lon_max = 150.57, 151.37

# Output directory
output_dir = "/g/data/gb02/mf9078/plots/final/paperFigs/"
os.makedirs(output_dir, exist_ok=True)

# Get list of files from first scenario (assuming all have same time steps)
first_scenario_path = list(paths.values())[0]
file_pattern = os.path.join(first_scenario_path, "wrfout_d02_2017-01-*")
reference_files = sorted(glob.glob(file_pattern))

print(f"Found {len(reference_files)} time steps to process")

# Process each time step
for file_idx, ref_file_path in enumerate(reference_files):
    try:
        # Extract filename for this time step
        filename = os.path.basename(ref_file_path)
        
        # Extract timestamp from filename and convert to AEST
        timestamp_utc = datetime.strptime(filename[11:], "%Y-%m-%d_%H:%M:%S")
        timestamp_aest = timestamp_utc + timedelta(hours=10)
        
        print(f"Processing time step {file_idx + 1}/{len(reference_files)}: {timestamp_aest.strftime('%Y-%m-%d %H:%M AEST')}")
        
        # Dictionary to store file paths for all scenarios
        scenario_files = {}
        data_dict = {}
        
        # Load data from all scenarios for this time step
        for scenario_name, scenario_path in paths.items():
            file_path = os.path.join(scenario_path, filename)
            
            if not os.path.exists(file_path):
                print(f"Warning: File not found: {file_path}")
                continue
                
            scenario_files[scenario_name] = file_path
            
        # Load datasets
        ds1 = xr.open_dataset(scenario_files["LCZ"])
        ds2 = xr.open_dataset(scenario_files["WSF-MB"])
        ds3 = xr.open_dataset(scenario_files["Geoscape"])
        
        # Get coordinates and select first time step (if there's a time dimension)
        XLAT = ds1["XLAT"].isel(Time=0) if "Time" in ds1["XLAT"].dims else ds1["XLAT"]
        XLONG = ds1["XLONG"].isel(Time=0) if "Time" in ds1["XLONG"].dims else ds1["XLONG"]
        LU_INDEX = ds1["LU_INDEX"].isel(Time=0) if "Time" in ds1["LU_INDEX"].dims else ds1["LU_INDEX"]
        
        # Create masks
        lat_mask = (XLAT >= lat_min) & (XLAT <= lat_max)
        lon_mask = (XLONG >= lon_min) & (XLONG <= lon_max)
        lu_mask = (LU_INDEX >= 51) & (LU_INDEX <= 60)
        region_mask = lat_mask & lon_mask & lu_mask
        
        # Get data and select first time step if needed for LCZ
        t2_data1 = ds1["T2"].isel(Time=0) if "Time" in ds1["T2"].dims else ds1["T2"]
        comf10_data1 = ds1["COMF_10"].isel(Time=0) if "Time" in ds1["COMF_10"].dims else ds1["COMF_10"]
        comf90_data1 = ds1["COMF_90"].isel(Time=0) if "Time" in ds1["COMF_90"].dims else ds1["COMF_90"]
        u10_data1 = ds1["U10"].isel(Time=0) if "Time" in ds1["U10"].dims else ds1["U10"]
        v10_data1 = ds1["V10"].isel(Time=0) if "Time" in ds1["V10"].dims else ds1["V10"]
       
        # Get data and select first time step if needed for WSF3DMSB
        t2_data2 = ds2["T2"].isel(Time=0) if "Time" in ds2["T2"].dims else ds2["T2"]
        comf10_data2 = ds2["COMF_10"].isel(Time=0) if "Time" in ds2["COMF_10"].dims else ds2["COMF_10"]
        comf90_data2 = ds2["COMF_90"].isel(Time=0) if "Time" in ds2["COMF_90"].dims else ds2["COMF_90"]
        u10_data2 = ds2["U10"].isel(Time=0) if "Time" in ds2["U10"].dims else ds2["U10"]
        v10_data2 = ds2["V10"].isel(Time=0) if "Time" in ds2["V10"].dims else ds2["V10"]
        
        # Get data and select first time step if needed for Geoscape
        t2_data3 = ds3["T2"].isel(Time=0) if "Time" in ds3["T2"].dims else ds3["T2"]
        comf10_data3 = ds3["COMF_10"].isel(Time=0) if "Time" in ds3["COMF_10"].dims else ds3["COMF_10"]
        comf90_data3 = ds3["COMF_90"].isel(Time=0) if "Time" in ds3["COMF_90"].dims else ds3["COMF_90"]
        u10_data3 = ds3["U10"].isel(Time=0) if "Time" in ds3["U10"].dims else ds3["U10"]
        v10_data3 = ds3["V10"].isel(Time=0) if "Time" in ds3["V10"].dims else ds3["V10"]
        
        # Apply masks
        t21 = t2_data1.where(region_mask) - 273.15
        comf101 = comf10_data1.where(region_mask)
        comf901 = comf90_data1.where(region_mask)
        u101 = u10_data1.where(region_mask)
        v101 = v10_data1.where(region_mask)
        
        t22 = t2_data2.where(region_mask) - 273.15
        comf102 = comf10_data2.where(region_mask)
        comf902 = comf90_data2.where(region_mask)
        u102 = u10_data2.where(region_mask)
        v102 = v10_data2.where(region_mask)
        
        t23 = t2_data3.where(region_mask) - 273.15
        comf103 = comf10_data3.where(region_mask)
        comf903 = comf90_data3.where(region_mask)
        u103 = u10_data3.where(region_mask)
        v103 = v10_data3.where(region_mask)
       
        # Plot
        variables = {
            "t2": {
                "LCZ": {"t2":t21, "u10":u101, "v10":v101},
                "WSF-MB": {"t2":t22, "u10":u102, "v10":v102},
                "Geoscape": {"t2":t23, "u10":u103, "v10":v103},
                "label": "2-m Air Temperature (°C)",
                "vmin": 20,
                "vmax": 40,
                "cmap": "jet",
                "norm": None
            },
            "comf10": {
                "LCZ": comf101,
                "WSF-MB": comf102,
                "Geoscape": comf103,
                "label": "UTCI Cool Spots (°C)",
                "vmin": 20,
                "vmax": 45,
                "cmap": utci_cmap,
                "norm": utci_norm
            },
            "comf90": {
                "LCZ": comf901,
                "WSF-MB": comf902,
                "Geoscape": comf903,
                "label": "UTCI Hot Spots (°C)",
                "vmin": 20,
                "vmax": 45,
                "cmap": utci_cmap,
                "norm": utci_norm
            } 
        }
        
        scs = ["LCZ", "WSF-MB", "Geoscape"]
        
        fig, axs = plt.subplots(3, 3, figsize=(20, 19), subplot_kw={"projection": ccrs.PlateCarree()}, 
                               sharey=True, sharex=True, constrained_layout=False)
        plt.subplots_adjust(bottom=0.015)  # Leave space for colorbar
        plt.tight_layout(h_pad=0.01) 
        
        # Add scenario labels to the left side of the first column
        scenario_label_positions = [0.8, 0.5, 0.2]  # Approximate y positions for each row
        for i, scenario in enumerate(scs):
            fig.text(-0.06, scenario_label_positions[i], scenario,
                     rotation=90, fontsize=36,
                     ha='center', va='center')

        # Add column titles at the top
        column_titles = ["2-m Air Temperature", "UTCI Cool Spots", "UTCI Hot Spots"]
        column_label_positions = [0.17, 0.5, 0.82]  # Approximate x positions for each column

        for i, title in enumerate(column_titles):
            fig.text(column_label_positions[i], 0.96, title,
                     rotation=0, fontsize=36, #fontweight='bold',
                     ha='center', va='top')
        
        for col, (varname, props) in enumerate(variables.items()):
            for sc in scs:
                row = scs.index(sc)
                ax = axs[row, col]
                ax.set_extent([lon_min, lon_max, lat_min, lat_max])
                ax.add_feature(cfeature.GSHHSFeature(scale="high"))
                ax.add_feature(cfeature.BORDERS, linestyle="--", edgecolor="gray")
                ax.add_feature(cfeature.LAND, facecolor="lightgray", alpha=0.3)

                if varname != "t2":
                    # comf90 and comf10
                    mesh = ax.pcolormesh(XLONG, XLAT, props[sc], transform=ccrs.PlateCarree(), 
                                         cmap=props["cmap"], norm=props["norm"], alpha=0.8, shading='nearest')
                else:
                    #t2
                    mesh = ax.pcolormesh(XLONG, XLAT, props[sc]["t2"], transform=ccrs.PlateCarree(), 
                                         cmap="jet", alpha=0.8, shading='nearest', 
                                         vmin=props["vmin"], vmax=props["vmax"])

                    skip = 4
                    Q=ax.quiver(
                        XLONG[::skip, ::skip], XLAT[::skip, ::skip], props[sc]["u10"][::skip, ::skip], props[sc]["v10"][::skip, ::skip], 
                        scale=10, scale_units='inches', color="black", width=0.0025
                    )

                    if row == 0:
                    #Optional: Add a quiver key (legend for wind speed)
                        qk = ax.quiverkey(Q, 0, 1.05, 2, r'$2 \frac{m}{s}$', labelpos='E',
                                           coordinates='axes', color='black', fontproperties={'size': 32})

                # Add location markers
                ax.plot(loc1_lon, loc1_lat, 'kx', markersize=8, markeredgewidth=2, 
                         transform=ccrs.PlateCarree(), zorder=5) #markerfacecolor='white',
                ax.plot(loc2_lon, loc2_lat, 'kx', markersize=8, markeredgewidth=2, 
                         transform=ccrs.PlateCarree(), zorder=5) #markerfacecolor='white',

                # Add colorbar
                if row == 2:
                    if varname in ["comf10", "comf90"]:
                        # For UTCI variables, use custom ticks at boundaries
                        ax1_pos = axs[2, 1].get_position()  # Bottom row, second column (comf10)
                        ax2_pos = axs[2, 2].get_position()  # Bottom row, third column (comf90)

                        # Create colorbar axis spanning both columns
                        cbar_ax = fig.add_axes([ax1_pos.x0+0.03, ax1_pos.y0 - 0.0365, 
                                                ax2_pos.x1 - ax1_pos.x0-0.06, 0.015])

                        # Create the shared UTCI colorbar
                        utci_cbar = plt.colorbar(mesh, cax=cbar_ax, orientation='horizontal')
                        # Set custom ticks and labels for UTCI colorbar
                        tick_locations = [23, 29, 35, 41, 47]
                        tick_labels = ['neutral', 'moderate', 'strong', 'very strong', 'extreme']
                        utci_cbar.set_ticks(tick_locations)
                        utci_cbar.set_ticklabels(tick_labels)
                        utci_cbar.ax.minorticks_off()
                        utci_cbar.ax.tick_params(size=0, labelsize=28)
                        utci_cbar.set_label("UTCI Stress Category", rotation=0, fontsize=30)

                    else:
                        # For temperature, use default colorbar
                        cbar = plt.colorbar(mesh, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8, extend='both')
                
                    cbar.set_label("Temperature (°C)", rotation=0, fontsize=30)
                    cbar.ax.tick_params(labelsize=28)
                    ax.set_xlabel("Longitude", fontsize=22)
                
                if col == 0:
                    ax.set_ylabel(f"{sc}\n\nLatitude", fontsize=24)
                
                # Add gridlines
                gl = ax.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False, alpha=0.2)
                gl.xlabel_style = {'size': 20}
                gl.ylabel_style = {'size': 20}
                gl.top_labels = False
                gl.right_labels = False
                
                # Only show y-axis labels (left labels) for the first column
                if col != 0:
                    gl.left_labels = False
                
                # Only show x-axis labels (bottom labels) for the bottom row
                if row != 2:
                    gl.bottom_labels = False
        
        # Set title with AEST time
        plt.suptitle(f"{timestamp_aest.strftime('%Y-%m-%d %H:%M')}", 
                     fontsize=36, y=0.99, fontweight='bold')
        
        # Save figure
        filename_out = f"hourly_heat_{timestamp_aest.strftime('%Y%m%d_%H%M')}_AEST.png"
        output_path = os.path.join(output_dir, filename_out)
        
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()  # Close to free memory
        
        # Close datasets
        ds1.close()
        ds2.close()
        ds3.close()
        
        print(f"Saved: {filename_out}")
        
    except Exception as e:
        print(f"Error processing file {ref_file_path}: {e}")
        continue

print("Processing complete!")

ERROR 1: PROJ: proj_create_from_database: Open of /g/data/xp65/public/./apps/med_conda/envs/analysis3-25.09/share/proj failed


Found 2 time steps to process
Processing time step 1/2: 2017-01-13 15:00 AEST
Saved: hourly_heat_20170113_1500_AEST.png
Processing time step 2/2: 2017-01-17 15:00 AEST
Saved: hourly_heat_20170117_1500_AEST.png
Processing complete!
